In [1]:
import pm4py

path = "data/scenario_test_ocel.json"  # example_3_ocel
ocel = pm4py.read_ocel2_json(path)

df_events = ocel.events.copy()
df_events.set_index("ocel:eid", inplace=True)
df_relations = ocel.relations.copy()
df_relations.set_index("ocel:eid", inplace=True)

df_events_objects = df_events.join(df_relations, rsuffix="_relations")

/home/mark/research/digital-factory-monitoring/.venv/lib/python3.11/site-packages/pm4py/utils.py:991: UserWarning: Install the optional requirement `rustxes` to import/export files faster.
  warnings.warn("Install the optional requirement `rustxes` to import/export files faster.")


In [2]:
import json
import networkx as nx


# Load the graphml and parse JSON attributes back
def load_graphml_with_json_attrs(path: str) -> nx.Graph:
    """Read a GraphML file and attempt to JSON-decode any string attributes back into Python objects.

    Only replaces attribute values when json.loads returns a dict or list (to avoid converting plain strings).
    Works for Graph/DiGraph and MultiGraph/MultiDiGraph edge representations.
    """
    G = nx.read_graphml(path)

    # Nodes
    for n, d in G.nodes(data=True):
        for k, v in list(d.items()):
            if isinstance(v, str):
                try:
                    parsed = json.loads(v)
                    if isinstance(parsed, (dict, list)):
                        d[k] = parsed
                except Exception:
                    # leave as string if it isn't JSON
                    pass

    # Edges (handle keyed MultiGraphs and non-keyed graphs)
    try:
        edges = list(G.edges(keys=True, data=True))
        keyed = True
    except TypeError:
        edges = list(G.edges(data=True))
        keyed = False

    if keyed:
        for u, v, key, ed in edges:
            for k, val in list(ed.items()):
                if isinstance(val, str):
                    try:
                        parsed = json.loads(val)
                        if isinstance(parsed, (dict, list)):
                            ed[k] = parsed
                    except Exception:
                        pass
    else:
        for u, v, ed in edges:
            for k, val in list(ed.items()):
                if isinstance(val, str):
                    try:
                        parsed = json.loads(val)
                        if isinstance(parsed, (dict, list)):
                            ed[k] = parsed
                    except Exception:
                        pass

    return G


graphml_path = path.replace(".json", ".graphml")
ocel_nx = load_graphml_with_json_attrs(graphml_path)

In [3]:
from collections import Counter

from process_execution import extract_process_execution

object_types = ["PackingUnit"]

events_to_trace = df_events_objects[
    (df_events_objects["ocel:type"].isin(object_types))
].index.values

print(f"Number of events selected: {len(events_to_trace)}")


def determine_class_quality(event: str):
    return ocel_nx.nodes()[event]["attr"].get("averageQuality") >= 1.0


def determine_class_attribute(trace_graph: nx.Graph):
    selected_activity = "Object-departing-WB"
    selected_attribute = "a"
    for _, data in trace_graph.nodes(data="attr"):
        if (
            data.get("ocel:activity", "") == selected_activity
            and data.get(selected_attribute, 1) < 0.25
        ):
            return False
    return True


trace_graphs = {}
for event in events_to_trace:
    trace_graph = extract_process_execution(
        ocel_nx,
        event,
        ["ProductionLot", "PackingUnit"],
        "Object-creating_class_instance",
    )
    trace_graph.construct_node_label()
    trace_graph.construct_edge_label()

    trace_graphs[event] = {
        "process_execution": trace_graph,
        # "class": determine_class_quality(event),
        "class": determine_class_attribute(trace_graph),
    }


Counter([d["class"] for d in trace_graphs.values()])

Number of events selected: 6250


Counter({True: 3786, False: 2464})

### Instance-based

Two step approach:
1) Find *k* graphs with different class, but similar structure (including node labels);
2) Among the *k* graphs, find the most similar graph, also considering the node attributes.

In [ ]:
from grakel.kernels import (
    VertexHistogram,
    WeisfeilerLehman,
)
from grakel.utils import graph_from_networkx

import numpy as np

k = 10  # select top k structurally most similar graphs

target_trace_graph_id = "100023"
target_trace_graph = trace_graphs[target_trace_graph_id]["process_execution"]

selected_trace_graphs = {
    k: trace_graphs[k]["process_execution"] for k in (list(trace_graphs.keys()))
}

gk = WeisfeilerLehman(n_iter=2, normalize=True, base_graph_kernel=VertexHistogram)

target_trace_graph_grakel = graph_from_networkx(
    [target_trace_graph],
    node_labels_tag="label",
    as_Graph=True,
    # val_node_labels="test",
)
gk.fit(target_trace_graph_grakel)

selected_trace_graphs_grakel = graph_from_networkx(
    selected_trace_graphs.values(),
    node_labels_tag="label",
    as_Graph=True,
    # val_node_labels="test",
)
K_gk = gk.transform(selected_trace_graphs_grakel)

most_similar_trace_graphs_gk = np.array(list(selected_trace_graphs.keys()))[
    np.argsort(K_gk[:, 0])[(-1 * k) :]
]
print(most_similar_trace_graphs_gk)

In [ ]:
import numpy as np

from grakel.kernels import SubgraphMatching

selected_trace_graphs = {
    k: nx.DiGraph(trace_graphs[k]["process_execution"])
    for k in most_similar_trace_graphs_gk
    if trace_graphs[k]["class"] != trace_graphs[target_trace_graph_id]["class"]
}


def numeric_diff(a, b):
    try:
        return 1 - (float(b) - float(a)) / float(a)
    except ZeroDivisionError:
        return 0.5
    except ValueError:
        return 0.5


def dict_compare(a, b):
    sim_score = 0
    for key in a.keys():
        v_a = a.get(key)
        v_b = b.get(key)

        if not (v_a and v_b):
            sim_score += 0

        try:
            sim_score += 1 - (float(v_b) - float(v_a)) / float(v_a)
        except ZeroDivisionError:
            sim_score += 0.5
        except TypeError:
            sim_score += int(v_a == v_b)
        except ValueError:
            sim_score += int(v_a == v_b)
    return sim_score


sub_match = SubgraphMatching(
    normalize=True,
    kv=dict_compare,
    ke=None,
)

target_trace_graph_grakel = graph_from_networkx(
    [target_trace_graph],
    node_labels_tag="attr",
    # as_Graph=True,
    # val_node_labels="test",
    # edge_labels_tag="attr",
)
sub_match.fit(target_trace_graph_grakel)

selected_trace_graphs_grakel = graph_from_networkx(
    selected_trace_graphs.values(),
    node_labels_tag="attr",
    # as_Graph=True,
    # val_node_labels="test",
    edge_labels_tag="attr",
)
K = sub_match.transform(selected_trace_graphs_grakel)

most_similar_trace_graph_id = list(selected_trace_graphs.keys())[
    np.argsort(K[:, 0])[-1]
]

print("Query process execution")
print("--------------")
print(target_trace_graph_id)
print()
print("Most similar process execution")
print("---------------------")
print(most_similar_trace_graph_id)

### Instance-based + optimization

Two step approach:
1) Find *k* graphs with different class, but similar structure (including node labels);
2) From the *k* similar graphs and optimize objective function by modifying node attributes.

In [4]:
from grakel.kernels import (
    VertexHistogram,
    WeisfeilerLehman,
)
from grakel.utils import graph_from_networkx

import numpy as np

k = 10  # select top k structurally most similar graphs

target_trace_graph_id = "100023"
target_trace_graph = trace_graphs[target_trace_graph_id]["process_execution"]

selected_trace_graphs = {
    k: trace_graphs[k]["process_execution"] for k in (list(trace_graphs.keys()))
}

gk = WeisfeilerLehman(n_iter=2, normalize=True, base_graph_kernel=VertexHistogram)

target_trace_graph_grakel = graph_from_networkx(
    [target_trace_graph],
    node_labels_tag="label",
    as_Graph=True,
    # val_node_labels="test",
)
gk.fit(target_trace_graph_grakel)

selected_trace_graphs_grakel = graph_from_networkx(
    selected_trace_graphs.values(),
    node_labels_tag="label",
    as_Graph=True,
    # val_node_labels="test",
)
K_gk = gk.transform(selected_trace_graphs_grakel)

most_similar_trace_graphs_gk = np.array(list(selected_trace_graphs.keys()))[
    np.argsort(K_gk[:, 0])[(-1 * k) :]
]
print(most_similar_trace_graphs_gk)

['44062' '449352' '448545' '448384' '444734' '444022' '443221' '442667'
 '449315' '99896']


In [27]:
import gnn_graph_classification

from importlib import reload

reload(gnn_graph_classification)

<module 'gnn_graph_classification' from '/home/mark/research/digital-factory-monitoring/src/analysis/gnn_graph_classification.py'>

In [ ]:
from gnn_graph_classification import (
    build_vocab_and_numeric_keys,
    convert_trace_graphs_to_pyg,
)

most_similar_trace_graphs = {i: trace_graphs[i] for i in most_similar_trace_graphs_gk}

# Modify node attribute value
most_similar_trace_graphs["449315"]["process_execution"].nodes()["448967"]["attr"]["a"] -= 0.4

node_label_vocab, node_num_keys, edge_num_keys = build_vocab_and_numeric_keys(
    trace_graphs
)
data_most_similar_trace_graphs = convert_trace_graphs_to_pyg(
    most_similar_trace_graphs, node_label_vocab, node_num_keys, edge_num_keys
)

In [34]:
# Generate predictions for the most similar trace graphs using the loaded model
import torch
from torch_geometric.loader import DataLoader as PyGLoader

from gnn_graph_classification import GCNWithEdgeAgg

device = "cuda" if torch.cuda.is_available() else "cpu"

# Ensure `model` is available — if it's a state dict, rebuild a model skeleton
try:
    model
except NameError:
    model = torch.load(path.replace(".json", "-model_weights.pth"), weights_only=False)

# If a state-dict was saved instead of a model object, try to instantiate the known architecture
if (
    isinstance(model, dict)
    and "state_dict" not in model
    and not hasattr(model, "__call__")
):
    # attempt to import the model class from our gnn module
    try:
        in_ch = data_most_similar_trace_graphs[0].x.shape[1]
        model_obj = GCNWithEdgeAgg(in_ch)
        model_obj.load_state_dict(model)
        model = model_obj
    except Exception as e:
        raise RuntimeError(
            "Loaded object appears to be a state-dict but could not instantiate model: "
            + str(e)
        )

model = model.to(device)
model.eval()

# Use a non-shuffling loader so order matches the graph id list
pred_loader = PyGLoader(data_most_similar_trace_graphs, batch_size=8, shuffle=False)

preds = []
probs = []
with torch.no_grad():
    for batch in pred_loader:
        batch = batch.to(device)
        # Model forward for PyG-style GNNs: (x, edge_index, batch)
        out = model(batch.x, batch.edge_index, batch.batch)
        pprob = torch.nn.functional.softmax(out, dim=-1)
        preds.extend(out.argmax(dim=-1).cpu().numpy().tolist())
        probs.extend(pprob.cpu().numpy().tolist())

# Map predictions back to graph ids (data_most_similar_trace_graphs preserves insertion order)
graph_ids = list(most_similar_trace_graphs.keys())
results = []
for gid, pred, prob in zip(graph_ids, preds, probs):
    results.append(
        {"graph_id": gid, "pred": int(pred), "probabilities": [float(x) for x in prob]}
    )

# Expose results to the notebook
predictions = results

# Print a short summary
for r in predictions:
    print(r["graph_id"], "-> pred=", r["pred"], "prob=", r["probabilities"])

44062 -> pred= 1 prob= [0.002600159728899598, 0.9973998069763184]
449352 -> pred= 1 prob= [0.0017015759367495775, 0.9982984662055969]
448545 -> pred= 1 prob= [0.0014546778984367847, 0.9985454082489014]
448384 -> pred= 1 prob= [0.0015165198128670454, 0.9984834790229797]
444734 -> pred= 0 prob= [0.9999207258224487, 7.924301462480798e-05]
444022 -> pred= 1 prob= [0.002546298783272505, 0.9974537491798401]
443221 -> pred= 1 prob= [0.0006066625937819481, 0.9993933439254761]
442667 -> pred= 1 prob= [0.0025061534252017736, 0.9974938631057739]
449315 -> pred= 0 prob= [1.0, 4.698310916885365e-15]
99896 -> pred= 0 prob= [0.9999974966049194, 2.552781552367378e-06]
